# Lecture 3 (Part 1): APIs and pandas 🐼

---

Data collection is exactly where AI helps most — and where unverified AI code fails most quietly.

## 3.0 Where financial data actually comes from

Until now we typed our data in. Real analysis starts with **getting data programmatically** — your group assignment will explicitly require it (you cannot submit data files, only code that fetches them). The mechanism is the **API** (Application Programming Interface): a set of rules by which your program requests data from someone's server and receives it in a structured format. You send a request ("monthly TSLA prices, 2012-2025"), the server answers with data.

The second half of the topic is what to *put* the data in: **pandas**, Python's data-analysis workhorse. Where NumPy gives you the array, pandas gives you the **Series** (one labeled column) and the **DataFrame** (a table with labeled rows and columns — think of a highly optimized Excel sheet): reading data, dates and time series, sorting, grouping, missing values.

One important skill: **never trust a download you haven't inspected.** Every dataset that enters your code gets four checks: *shape* (how many rows/columns?), *date range* (does it cover what you asked?), *missing values* (where, and why?), and *a plot* (do the numbers look like prices/returns, or like garbage?).


In [ ]:
import numpy as np
import pandas as pd

---
## 3.1 The FRED API: 800,000 economic time series

FRED (the St. Louis Fed's database) offers a free API — you request a series by its code, you get it as a pandas object. You need a (free) API key from https://fred.stlouisfed.org/docs/api/api_key.html — **get your own**; it takes two minutes, and it is your identity towards the server.

In [ ]:
!pip install -q fredapi
from fredapi import Fred
import os
fred = Fred(api_key=os.environ.get('FRED_API_KEY', 'PASTE_YOUR_KEY_HERE'))   # set FRED_API_KEY, or paste your own key here
data = fred.get_series('UNRATE')          # US unemployment rate
print(type(data))
data.head() # preview

### Pandas `Series`

What came back is a `Series`: one column of data with a labeled **index** — here, dates. A `Series` behaves like a NumPy array that *knows what its entries mean*. You can also build one yourself:

In [ ]:
s = pd.Series(np.random.normal(0,0.02,5), name='daily returns') # generate 5 random returns
s.index = ['AMZN', 'AAPL', 'MSFT', 'GOOG', 'TSLA']
print(s['MSFT'])       # select by label
print(s.iloc[2])       # select by position -- .iloc
s

### Discussion

`data.describe()` summarizes a Series. Its *output* is itself a Series — with what index? And what will `sum_stats['max']` return?

In [ ]:
sum_stats = data.describe()
print(sum_stats)
print(f"The median unemployment rate is {sum_stats['50%']:.2f}%")
print(f"The maximum unemployment rate is {sum_stats['max']:.2f}%")
print('std' in sum_stats)    # you can even ask whether an index is present

And the next check — **the plot**


In [ ]:
ax = data['2006':'2026'].plot(title='US Unemployment Rate', legend=False)
ax.set_xlabel('year', fontsize=12)
ax.set_ylabel('%', fontsize=12)

---
## 3.2 Market data from Yahoo Finance

In [ ]:
import yfinance as yf

df = yf.download("TSLA",
                 start="2012-01-01",
                 end="2025-12-31",
                 interval='1mo',
                 auto_adjust = True,
                 progress=False)
print(f"Downloaded {len(df)} rows of data.")
print(type(df))
df

Options worth knowing: a **list of tickers** downloads many at once; `auto_adjust=True` gives split/dividend-adjusted prices (the ones returns must be computed from); `interval` goes from `'1m'` (intraday) to `'3mo'`; `actions='inline'` adds dividends and splits; `progress=False` silences the progress bar.

Note: the output comes as MutiIndex Columns, that is, each column comes with two headers ('Close' + 'TSLA')

In [ ]:
# 4 checks
# 1: shape
print("Shape of the data", df.shape)
# 2: date range
print(f"Start date: {df.index[0]:%B %d, %Y}, End date: {df.index[-1]:%B %d, %Y}")
# 3: missing values
print("Missing values \n", df.isna().sum())
# 4: plot
df.Close.plot()


### DataFrames: selecting data

A `DataFrame` is several labeled columns. Selecting subsets is an important pandas skill. Write down what each line returns (rows? columns? both?), then run:

In [ ]:
tesla_data = yf.Ticker("TSLA")
df_daily = tesla_data.history()        # recent daily price history
df_daily

In [ ]:
df_daily[2:5]                          # 1. slicing with numbers -> ?

In [ ]:
df_daily[['High', 'Low']]              # 2. a list of names -> ?

In [ ]:
df_daily.iloc[2:5, 0:4]                # 3. .iloc[rows, columns] -- by position

In [ ]:
df_daily.loc[df_daily.index[2:5], ['High', 'Low']]   # 4. .loc -- by label

In [ ]:
df_daily.head()               # 5. What does df_daily.tail() do?

### Selecting by condition — the one that matters

This is boolean indexing from last week, applied to tables:

In [ ]:
df_daily[df_daily.Open > df_daily.Open.median()]                 # rows where the open exceeded the median

In [ ]:
df_daily[(df_daily.Close - df_daily.Open > 0) & (df_daily.Open > df_daily.Open.median())] # conditions combine with & and |

### The `apply` method

`df.apply(f)` applies a function to each column (or row) and collects the results:

In [ ]:
df_daily[['Close', 'Volume']].apply(max)

*(More pandas: the suggested DataCamp course, or https://pandas.pydata.org/docs/. From here on, pandas questions are also excellent AI-explainer questions.)*

### Explore: Company information

The `Ticker` object carries far more than prices — `info` (profile and metrics), `actions` (dividends/splits), `institutional_holders`, `financials`, `quarterly_financials`, `calendar`... See the [yfinance documentation](https://ranaroussi.github.io/yfinance).

In [ ]:
tesla_data = yf.Ticker("TSLA")
print(tesla_data.info['website'])
tesla_data.institutional_holders

In [ ]:
tesla_data.quarterly_financials

---
## 3.3 Practice

Your groupmate wrote this cell to build the assignment dataset: monthly prices for four stocks since 2015, cleaned of missing values. It runs without errors. **Run it, then inspect it. What went wrong, and on which line?**

In [ ]:
# --- groupmate's code: runs fine, and yet... ---------------------------
tickers = ['AAPL', 'MSFT', 'NVDA', 'PLTR']
prices = yf.download(tickers, start='2015-01-01', end='2024-12-31',
                     interval='1mo', progress=False, auto_adjust=True)['Close']
prices = prices.dropna()                          # "clean the data"
rets = np.log(prices).diff(1).dropna()

print(prices.shape)
print(rets.index.min(), "->", rets.index.max())

<details>
<summary><b>Solution</b></summary>

Palantir only listed in **September 2020**. Before that date its column is all NaN — so `dropna()`, which removes any row containing a missing value, silently deleted **five years of data for all four stocks**. The sample now starts in 2020, and you goal to get the data "since 2015" was not reached. No error was ever raised.

The checks catch it instantly: `prices.shape` has far fewer rows than ~120, and the date range starts in 2020. Fixes depend on what you actually want: keep the shorter common sample *knowingly*; or drop the short-history ticker; or use `dropna(how='all')` and handle each stock's own history. What is not acceptable is not knowing which one you did. Note the family resemblance to survivorship bias: **data-cleaning choices silently define your sample.** Your group assignment's verification section should state the sample your cleaning produced.

</details>

---
## 3.4 URL requests: when there is no package

Behind every API is a URL. The `requests` library fetches anything; `pd.read_csv` can even read a CSV **directly from a URL** (find it on the FRED page: Download → CSV, copy link):

In [ ]:
import requests

url = ('https://fred.stlouisfed.org/graph/fredgraph.csv?id=UNRATE'
       '&cosd=1948-01-01&coed=2024-10-01&fq=Monthly&fam=avg')
source = requests.get(url).content.decode().split("\n")
print(source[0])     # the header line of the CSV
print(source[1])     # the first data line

In [ ]:
data = pd.read_csv(url, index_col=0, parse_dates=True)
print(type(data))
data.head()

---
## 3.5 Homework: first prompt problem

**The task:** produce a DataFrame of *monthly log returns* (not prices) for EDP, Galp, and Jerónimo Martins — three Lisbon-listed names — from 2018 through 2024, with no missing values, and report each stock's annualized volatility.

1. **Specify:** write a prompt for your AI assistant https://ai.novasbe.pt. Include everything a smart colleague would need: tickers *as Yahoo knows them* (Lisbon listings carry a suffix — find it), adjusted prices, the return definition, the date range, the cleaning rule, the annualization convention.
2. **Generate:** run the prompt, paste the code here, run it.
3. **Verify:** run the four checks. Are the volatilities *plausible* for these stocks? What annualized volatility would you expect for a utility vs. an oil company?



In [ ]:
# solution


<details>
<summary><b>Discussion, not solution</b></summary>

Things that separate good prompts here: Yahoo's Lisbon suffix (`EDP.LS`, `GALP.LS`, `JMT.LS`); saying *log* returns and *monthly* explicitly; `auto_adjust=True`; stating the annualization factor; saying what `dropna` should do. The lesson is not that AI is bad at this — it is usually excellent — but that **the ambiguity you leave in the prompt comes back as silent assumptions in the code.** Specification *is* the skill.

</details>



---
## 3.6 Explore: crypto data from CoinGecko

Another API, same pattern — request, receive, put in a DataFrame:

In [ ]:
!pip install -q pycoingecko
from pycoingecko import CoinGeckoAPI
cg = CoinGeckoAPI()

# Bitcoin OHLC (open-high-low-close) prices, last 14 days
ohlc = cg.get_coin_ohlc_by_id(id="bitcoin", vs_currency="usd", days="14")
ohlc_df = pd.DataFrame(ohlc)
ohlc_df.columns = ["date", "open", "high", "low", "close"]
ohlc_df["date"] = pd.to_datetime(ohlc_df["date"], unit="ms")
ohlc_df

In [ ]:
# current prices, market cap, 24h volume...
cg.get_price(ids=["bitcoin", "ethereum"], vs_currencies=["usd", "eur"])